# Análise Exploratória — Camada Bronze


In [0]:

from pyspark.sql import functions as F

df = spark.table("ifood_case.bronze.yellow_taxi_trips")

df.select(
    F.count("*").alias("total"),
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("passenger_null"),
    F.sum((F.col("total_amount") <= 0).cast("int")).alias("amount_nao_positivo"),
    F.sum((F.col("tpep_pickup_datetime") < "2023-01-01").cast("int")).alias("antes_jan"),
    F.sum((F.col("tpep_pickup_datetime") >= "2023-06-01").cast("int")).alias("depois_mai"),
).display()

In [0]:
def to_silver(table: str, taxi_type: str, pickup: str, dropoff: str):
    return (
        spark.table(table)
        .withColumnRenamed(pickup, "pickup_datetime")
        .withColumnRenamed(dropoff, "dropoff_datetime")
        .withColumn("taxi_type", F.lit(taxi_type))
        .filter(
            (F.col("pickup_datetime") >= "2023-01-01") &
            (F.col("pickup_datetime") <  "2023-06-01") &
            (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
            (F.col("total_amount") > 0)
        )
        .dropDuplicates(["VendorID", "pickup_datetime", "dropoff_datetime",
                         "passenger_count", "total_amount"])
        .select("VendorID", "passenger_count", "total_amount",
                "pickup_datetime", "dropoff_datetime", "taxi_type")
    )

silver = (
    to_silver("ifood_case.bronze.yellow_taxi_trips", "yellow",
              "tpep_pickup_datetime", "tpep_dropoff_datetime")
    .unionByName(
        to_silver("ifood_case.bronze.green_taxi_trips", "green",
                  "lpep_pickup_datetime", "lpep_dropoff_datetime"))
)

(silver.write.format("delta").mode("overwrite")
    .partitionBy("taxi_type")
    .saveAsTable("ifood_case.silver.taxi_trips"))

In [0]:
%sql
CREATE OR REPLACE VIEW ifood_case.silver.yellow_taxi_trips AS
SELECT
  VendorID,
  passenger_count,
  total_amount,
  pickup_datetime  AS tpep_pickup_datetime,
  dropoff_datetime AS tpep_dropoff_datetime
FROM ifood_case.silver.taxi_trips
WHERE taxi_type = 'yellow';

In [0]:
%sql
SHOW TABLES IN ifood_case.silver;

In [0]:
%sql
SELECT 'bronze_yellow' AS tabela, COUNT(*) AS total FROM ifood_case.bronze.yellow_taxi_trips
UNION ALL
SELECT 'bronze_green', COUNT(*) FROM ifood_case.bronze.green_taxi_trips
UNION ALL
SELECT 'silver_total', COUNT(*) FROM ifood_case.silver.taxi_trips
UNION ALL
SELECT 'silver_yellow_view', COUNT(*) FROM ifood_case.silver.yellow_taxi_trips;

## Conclusões da EDA
- A base bruta tem [X]% de registros com total_amount <= 0 (estornos/ajustes) e [Y]% fora do range Jan-Mai/2023 — ambos removidos na camada silver.
- A distribuição de total_amount mostra [descreva o que você observar: cauda longa, outliers, etc].
- Volume mensal é [estável / crescente / etc ao longo do período].